# Metrics HasAns NoAns và answerability trên dev

Notebook này tính EM/F1 HasAns/NoAns cho các dự đoán đã lưu và tính accuracy/confusion matrix no-answer cho từng phương pháp. Không huấn luyện lại mô hình và không đọc test.json. Nhãn thật lấy từ dev.json; dự đoán ghép theo id. Kết quả được lưu trong results/dev_answerability_and_group_metrics.json.


## Bước 1 — Nạp dev và hàm metric dùng chung


In [1]:
import csv
import json
import sys
from pathlib import Path

# Xác định thư mục gốc dù kernel chạy từ notebooks hay thư mục dự án.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

# Cho phép import hàm đánh giá dùng trong pipeline fine-tune.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluation import score_prediction

# Mỗi dòng JSONL là một mẫu dev.
dev_path = project_root / "data" / "processed" / "dev.json"
dev_examples = []
with dev_path.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            dev_examples.append(json.loads(line))

# Tra cứu mẫu chuẩn theo id để ghép chính xác.
dev_by_id = {example["id"]: example for example in dev_examples}
dev_ids = set(dev_by_id)
if len(dev_by_id) != len(dev_examples):
    raise ValueError("dev.json có id bị lặp.")

print("Dev:", len(dev_examples))
print("HasAns:", sum(not row["is_impossible"] for row in dev_examples))
print("NoAns:", sum(bool(row["is_impossible"]) for row in dev_examples))


Dev: 3814
HasAns: 2653
NoAns: 1161


## Bước 2 — Đọc predictions đã lưu
Jaccard nằm trong CSV; các phương pháp khác nằm trong JSON. Mỗi file phải có đúng một dự đoán cho từng id trong dev. Nếu sau này có predictions PhoBERT, notebook tự thêm các seed đó.


In [2]:
def read_json_predictions(path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

def read_csv_predictions(path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))

prediction_files = {
    "Jaccard": project_root / "results" / "baseline_predictions.csv",
    "TF-IDF + Logistic Regression": project_root / "results" / "baseline_tfidf_logreg_predictions.json",
    "Zero-shot": project_root / "results" / "zeroshot_predictions.json",
}

# Thêm từng seed fine-tune đã có.
for seed in [42, 123, 1024]:
    prediction_files[f"XLM-R seed {seed}"] = (
        project_root / "results" / "finetune" / "xlm-roberta"
        / f"seed-{seed}" / "predictions.json"
    )
    prediction_files[f"mDeBERTa seed {seed}"] = (
        project_root / "results" / "finetune" / "mdeberta"
        / f"seed-{seed}" / "predictions.json"
    )

# Nếu đã có dự đoán PhoBERT thì tự thêm vào danh sách.
phobert_dir = project_root / "results" / "finetune" / "phobert"
for path in sorted(phobert_dir.glob("seed-*/predictions.json")):
    seed = path.parent.name.removeprefix("seed-")
    prediction_files[f"PhoBERT seed {seed}"] = path

# Đọc từng file; nếu file bắt buộc thiếu, thông báo rõ đường dẫn.
predictions_by_method = {}
for method, path in prediction_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file dự đoán: {path}")

    if path.suffix.lower() == ".csv":
        rows = read_csv_predictions(path)
    else:
        rows = read_json_predictions(path)

    ids = [row["id"] for row in rows]
    id_set = set(ids)
    if len(id_set) != len(ids):
        raise ValueError(f"{method} có id dự đoán bị lặp.")

    if id_set != dev_ids:
        missing = dev_ids - id_set
        extra = id_set - dev_ids
        raise ValueError(
            f"{method}: id không khớp dev; thiếu {len(missing)}, "
            f"thừa {len(extra)}."
        )

    predictions_by_method[method] = rows
    print(f"{method}: {len(rows)} dự đoán; id khớp dev=True")


Jaccard: 3814 dự đoán; id khớp dev=True
TF-IDF + Logistic Regression: 3814 dự đoán; id khớp dev=True
Zero-shot: 3814 dự đoán; id khớp dev=True
XLM-R seed 42: 3814 dự đoán; id khớp dev=True
mDeBERTa seed 42: 3814 dự đoán; id khớp dev=True
XLM-R seed 123: 3814 dự đoán; id khớp dev=True
mDeBERTa seed 123: 3814 dự đoán; id khớp dev=True
XLM-R seed 1024: 3814 dự đoán; id khớp dev=True
mDeBERTa seed 1024: 3814 dự đoán; id khớp dev=True


## Bước 3 — Tính HasAns/NoAns EM/F1 và confusion matrix
HasAns nghĩa là nhãn thật is_impossible=False; NoAns nghĩa là True. Dự đoán NoAns được xác định bằng prediction_text == chuỗi rỗng. Confusion matrix có hàng là nhãn thật, cột là nhãn dự đoán, theo thứ tự HasAns rồi NoAns.


In [3]:
results = {}

for method, rows in predictions_by_method.items():
    # Dùng id thay vì trông vào thứ tự dòng của file.
    prediction_by_id = {row["id"]: row for row in rows}

    scores = {
        "HasAns": {"em": [], "f1": []},
        "NoAns": {"em": [], "f1": []},
    }

    # Bốn ô ma trận khi coi NoAns là lớp dương.
    true_has_pred_has = 0
    true_has_pred_no = 0
    true_no_pred_has = 0
    true_no_pred_no = 0

    for example_id, gold in dev_by_id.items():
        prediction_text = prediction_by_id[example_id].get("prediction_text", "")
        is_noans_true = bool(gold["is_impossible"])
        is_noans_pred = prediction_text == ""

        group = "NoAns" if is_noans_true else "HasAns"
        em, f1 = score_prediction(
            prediction_text,
            gold["answers"].get("text", []),
        )
        scores[group]["em"].append(em)
        scores[group]["f1"].append(f1)

        # Cập nhật confusion matrix: hàng nhãn thật, cột dự đoán.
        if not is_noans_true and not is_noans_pred:
            true_has_pred_has += 1
        elif not is_noans_true and is_noans_pred:
            true_has_pred_no += 1
        elif is_noans_true and not is_noans_pred:
            true_no_pred_has += 1
        else:
            true_no_pred_no += 1

    # Tính điểm trung bình theo nhóm và đổi sang phần trăm.
    group_metrics = {}
    for group, values in scores.items():
        count = len(values["em"])
        group_metrics[group] = {
            "count": count,
            "EM_percent": 100 * sum(values["em"]) / count,
            "F1_percent": 100 * sum(values["f1"]) / count,
        }

    # Hàng: nhãn thật HasAns/NoAns; cột: dự đoán HasAns/NoAns.
    matrix = [
        [true_has_pred_has, true_has_pred_no],
        [true_no_pred_has, true_no_pred_no],
    ]
    total = len(dev_examples)
    accuracy = 100 * (true_has_pred_has + true_no_pred_no) / total
    noans_total = true_no_pred_has + true_no_pred_no
    noans_recall = 100 * true_no_pred_no / noans_total if noans_total else 0.0

    results[method] = {
        "groups": group_metrics,
        "answerability_accuracy_percent": accuracy,
        "no_answer_recall_percent": noans_recall,
        "confusion_matrix_labels": ["HasAns", "NoAns"],
        "confusion_matrix": matrix,
    }

    print()
    print(method)
    for group in ["HasAns", "NoAns"]:
        metric = group_metrics[group]
        print(
            f"{group}: {metric['count']} mẫu | "
            f"EM={metric['EM_percent']:.2f}% | "
            f"F1={metric['F1_percent']:.2f}%"
        )
    print(f"Accuracy answerability: {accuracy:.2f}%")
    print(f"Recall NoAns: {noans_recall:.2f}%")
    print("Confusion matrix [thật HasAns/NoAns, dự đoán HasAns/NoAns]:")
    print(matrix)



Jaccard
HasAns: 2653 mẫu | EM=1.62% | F1=32.32%
NoAns: 1161 mẫu | EM=0.00% | F1=0.00%
Accuracy answerability: 69.56%
Recall NoAns: 0.00%
Confusion matrix [thật HasAns/NoAns, dự đoán HasAns/NoAns]:
[[2653, 0], [1161, 0]]

TF-IDF + Logistic Regression
HasAns: 2653 mẫu | EM=1.06% | F1=23.03%
NoAns: 1161 mẫu | EM=23.43% | F1=23.43%
Accuracy answerability: 58.36%
Recall NoAns: 23.43%
Confusion matrix [thật HasAns/NoAns, dự đoán HasAns/NoAns]:
[[1954, 699], [889, 272]]

Zero-shot
HasAns: 2653 mẫu | EM=39.65% | F1=58.93%
NoAns: 1161 mẫu | EM=45.65% | F1=45.65%
Accuracy answerability: 68.59%
Recall NoAns: 45.65%
Confusion matrix [thật HasAns/NoAns, dự đoán HasAns/NoAns]:
[[2086, 567], [631, 530]]

XLM-R seed 42
HasAns: 2653 mẫu | EM=43.87% | F1=59.32%
NoAns: 1161 mẫu | EM=32.04% | F1=32.04%
Accuracy answerability: 64.24%
Recall NoAns: 32.04%
Confusion matrix [thật HasAns/NoAns, dự đoán HasAns/NoAns]:
[[2078, 575], [789, 372]]

mDeBERTa seed 42
HasAns: 2653 mẫu | EM=0.00% | F1=0.55%
NoAns: 116

## Bước 4 — Lưu kết quả
Accuracy là tỷ lệ đúng trên cả hai lớp. Recall NoAns cho biết tỷ lệ câu no-answer thật sự được nhận ra. Ma trận được in theo hàng thật [HasAns, NoAns] và cột dự đoán [HasAns, NoAns]; có thể đọc lần lượt là TN, FP, FN, TP khi coi NoAns là lớp dương.


In [4]:
output = {
    "split": "dev",
    "sample_count": len(dev_examples),
    "no_answer_rule": "prediction_text == empty string",
    "confusion_matrix_order": {
        "rows": ["true_HasAns", "true_NoAns"],
        "columns": ["predicted_HasAns", "predicted_NoAns"],
    },
    "methods": results,
}

output_path = (
    project_root / "results" / "dev_answerability_and_group_metrics.json"
)
with output_path.open("w", encoding="utf-8") as file:
    json.dump(output, file, ensure_ascii=False, indent=2)

print("Đã lưu kết quả dev:", output_path)


Đã lưu kết quả dev: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\dev_answerability_and_group_metrics.json


## Kết quả kiểm tra cần thấy
Dev phải có 3.814 mẫu, gồm 2.653 HasAns và 1.161 NoAns. Mỗi phương pháp phải báo 3.814 dự đoán và ID khớp dev=True. Kết quả gồm HasAns/NoAns EM/F1, answerability accuracy, Recall NoAns và confusion matrix. Đây là kết quả dev, không phải test.
